# Búsqueda responsable de empresas en SEPREC

Este notebook procesa una lista de empresas usando la API pública observada en el portal de SEPREC. Está diseñado para unas 200 búsquedas diarias con pausas, caché persistente y reanudación.

**Importante:** no incluye cookies ni credenciales. Si el endpoint exige una sesión vigente, pega tu propio valor temporal en `SEPREC_COOKIE` durante la sesión y no guardes ni compartas el notebook con ese valor. No se intenta evadir Cloudflare, límites de tasa ni controles de acceso.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import random
import time
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any

import requests

BASE_URL = "https://servicios.seprec.gob.bo/api/empresas/buscarEmpresas"
TIPO_FILTRO = "nombre"
RESULTS_PER_PAGE = 10
DAILY_REQUEST_LIMIT = 200
MIN_DELAY_SECONDS = 8.0
MAX_DELAY_SECONDS = 18.0
MAX_RETRIES = 3
CACHE_PATH = Path("seprec_cache.json")
EXPORT_JSON_PATH = Path("seprec_resultados.json")
EXPORT_CSV_PATH = Path("seprec_resultados.csv")

# Proporciona el valor temporal desde una variable de entorno, nunca lo guardes en el notebook.
SEPREC_COOKIE = os.getenv("SEPREC_COOKIE", "").strip()

HEADERS = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "en-BO,en;q=0.9,es-BO;q=0.8,es;q=0.7",
    "origin": "https://miempresa.seprec.gob.bo",
    "referer": "https://miempresa.seprec.gob.bo/",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/156.0.0.0 Safari/537.36",
}

session = requests.Session()
session.headers.update(HEADERS)
if SEPREC_COOKIE:
    session.cookies.set("cf_clearance", SEPREC_COOKIE, domain="servicios.seprec.gob.bo")

print(f"Cookie configurada: {'sí' if bool(SEPREC_COOKIE) else 'no'}")

## 1. Lista de empresas

Reemplaza la lista de ejemplo por tus nombres. También puedes cargar un archivo `empresas.txt`, una empresa por línea.

In [ ]:
INPUT_PATH = Path("empresas.csv")


def load_company_names(input_path: Path) -> list[str]:
    with input_path.open(newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file, skipinitialspace=True)
        if not reader.fieldnames or "EMPRESA" not in reader.fieldnames:
            raise ValueError(
                "El CSV debe contener una columna llamada EMPRESA. "
                f"Columnas encontradas: {reader.fieldnames}"
            )
        return list(
            dict.fromkeys(
                row["EMPRESA"].strip().upper()
                for row in reader
                if row.get("EMPRESA") and row["EMPRESA"].strip()
            )
        )


if not INPUT_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {INPUT_PATH}")

EMPRESAS = load_company_names(INPUT_PATH)
if len(EMPRESAS) > DAILY_REQUEST_LIMIT:
    raise ValueError(
        f"La lista tiene {len(EMPRESAS)} empresas; "
        f"el límite configurado es {DAILY_REQUEST_LIMIT}."
    )
print(f"Empresas cargadas desde {INPUT_PATH}: {len(EMPRESAS)}")
print(EMPRESAS)

In [ ]:
# Prueba local: confirma que el CSV se carga antes de consumir la API.
assert EMPRESAS == [
    "HIPERMAXI",
    "ABRAHAN TITO MAMANI",
    "MI ALIMENTITO",
]
print("Prueba de carga CSV correcta: 3 empresas listas para consultar.")

In [ ]:
def load_cache() -> dict[str, Any]:
    if not CACHE_PATH.exists():
        return {"date": str(date.today()), "request_count": 0, "results": {}}
    cache = json.loads(CACHE_PATH.read_text(encoding="utf-8"))
    if cache.get("date") != str(date.today()):
        return {"date": str(date.today()), "request_count": 0, "results": cache.get("results", {})}
    return cache


def save_cache(cache: dict[str, Any]) -> None:
    temporary_path = CACHE_PATH.with_suffix(".tmp")
    temporary_path.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary_path.replace(CACHE_PATH)


cache = load_cache()
print(f"Peticiones usadas hoy: {cache['request_count']}/{DAILY_REQUEST_LIMIT}")
print(f"Resultados almacenados: {len(cache['results'])}")

## 2. Cliente con pausas y reintentos

Cada empresa se consulta como máximo una vez por ejecución si ya existe en la caché. Los errores `429` y `5xx` esperan con retroceso exponencial; un `403` se registra y detiene el proceso para evitar insistir ante un bloqueo.

In [ ]:
def wait_between_requests() -> None:
    delay = random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS)
    print(f"Pausa de {delay:.1f} segundos...")
    time.sleep(delay)


def request_company(company_name: str) -> dict[str, Any]:
    params = {
        "filtro": company_name,
        "tipoFiltro": TIPO_FILTRO,
        "limite": RESULTS_PER_PAGE,
        "pagina": 1,
    }
    for attempt in range(MAX_RETRIES + 1):
        try:
            response = session.get(BASE_URL, params=params, timeout=(10, 30))
        except requests.RequestException as exc:
            if attempt >= MAX_RETRIES:
                return {"status": "network_error", "error": str(exc)}
            backoff = 2 ** attempt * 10 + random.uniform(0, 5)
            print(f"Error de red; reintento en {backoff:.1f} segundos")
            time.sleep(backoff)
            continue

        if response.status_code == 403:
            return {"status": "forbidden", "http_status": 403, "error": "Acceso rechazado; detener para no insistir."}
        if response.status_code == 429 or response.status_code >= 500:
            if attempt >= MAX_RETRIES:
                return {"status": "http_error", "http_status": response.status_code, "error": response.text[:500]}
            retry_after = response.headers.get("Retry-After")
            backoff = float(retry_after) if retry_after and retry_after.isdigit() else 2 ** attempt * 30
            backoff += random.uniform(0, 10)
            print(f"HTTP {response.status_code}; reintento en {backoff:.1f} segundos")
            time.sleep(backoff)
            continue

        try:
            response.raise_for_status()
            return {"status": "ok", "http_status": response.status_code, "data": response.json()}
        except (requests.HTTPError, ValueError) as exc:
            return {"status": "invalid_response", "http_status": response.status_code, "error": str(exc), "body": response.text[:500]}
    return {"status": "unknown_error"}

In [ ]:
pending = [company for company in EMPRESAS if company not in cache["results"]]
available_requests = DAILY_REQUEST_LIMIT - int(cache["request_count"])
if len(pending) > available_requests:
    print(f"Solo se procesarán {available_requests} empresas hoy; quedan {len(pending) - available_requests} para otra ejecución.")
    pending = pending[:max(available_requests, 0)]

for index, company in enumerate(pending, start=1):
    if cache["request_count"] >= DAILY_REQUEST_LIMIT:
        break
    if index > 1:
        wait_between_requests()
    print(f"[{index}/{len(pending)}] Buscando: {company}")
    cache["results"][company] = {
        "queried_at": datetime.now(timezone.utc).isoformat(),
        **request_company(company),
    }
    cache["request_count"] += 1
    save_cache(cache)
    if cache["results"][company]["status"] == "forbidden":
        print("El servidor rechazó la solicitud; se detiene el lote.")
        break

print(f"Terminada la sesión. Peticiones usadas hoy: {cache['request_count']}/{DAILY_REQUEST_LIMIT}")

In [ ]:
EXPORT_JSON_PATH.write_text(json.dumps(cache["results"], ensure_ascii=False, indent=2), encoding="utf-8")

with EXPORT_CSV_PATH.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["empresa", "status", "http_status", "queried_at", "data", "error"])
    writer.writeheader()
    for company, result in cache["results"].items():
        writer.writerow({
            "empresa": company,
            "status": result.get("status", ""),
            "http_status": result.get("http_status", ""),
            "queried_at": result.get("queried_at", ""),
            "data": json.dumps(result.get("data", ""), ensure_ascii=False),
            "error": result.get("error", ""),
        })

print(f"Exportados: {EXPORT_JSON_PATH} y {EXPORT_CSV_PATH}")